# Supplied national climate tables: scoped audit
Run from research/discovery. Input files are unmodified copies of the user attachments. Saved official HTML was retrieved on 11 September 2026 (rainfall form region1=WHI; temperature mean series). This checks numeric transcription and structure, not scientific validity or homogeneity. The rainfall discrepancy is retained, not repaired.

In [1]:
import csv, math, statistics, re, html, json
from pathlib import Path
data_dir = Path("evidence/pasted-climate-check")
results = {}
for kind in ["rainfall", "temperature"]:
    rows = list(csv.DictReader((data_dir / (kind + ".csv")).open()))
    years = [int(r["Year"]) for r in rows]
    measures = list(rows[0])[1:18]
    numeric = [float(r[c]) for r in rows for c in measures]
    raw = (data_dir / (kind + "-selected.html")).read_text()
    cells = [html.unescape(re.sub("<[^>]*>", "", x)).strip()
             for x in re.findall(r"<t[dh]\b[^>]*>(.*?)</t[dh]>", raw, re.S | re.I)]
    source = {}
    for i, cell in enumerate(cells):
        if re.fullmatch(r"(19|20)\d{2}", cell) and i + 17 < len(cells):
            try: values = [float(x) for x in cells[i+1:i+18]]
            except ValueError: continue
            source.setdefault(cell, values)
    differences = [r["Year"] for r in rows if source.get(r["Year"]) != [float(r[c]) for c in measures]]
    result = {"rows": len(rows), "columns": len(rows[0]), "year_range": [min(years), max(years)],
              "duplicate_years": len(years)-len(set(years)),
              "missing_years": sorted(set(range(1901,2025))-set(years)),
              "empty_cells": sum(v == "" for r in rows for v in r.values()),
              "nonfinite_values": sum(not math.isfinite(v) for v in numeric),
              "source_comparison_different_years": differences,
              "source_numeric_values_compared": len(rows)*len(measures)}
    if kind == "rainfall":
        months = measures[:12]
        discrepancies = [{"year": int(r["Year"]), "annual_minus_months_mm": round(float(r["Annual"])-sum(float(r[m]) for m in months), 5)} for r in rows]
        result["annual_sum_discrepancies"] = discrepancies
        result["annual_discrepancies_above_0_65mm"] = sum(abs(x["annual_minus_months_mm"]) > 0.65 for x in discrepancies)
        result["2024_month_sum_mm"] = sum(float(rows[-1][m]) for m in months)
        result["2024_annual_mm"] = float(rows[-1]["Annual"])
    else:
        baseline = statistics.mean(float(r["Annual"]) for r in rows if 1991 <= int(r["Year"]) <= 2020)
        result["baseline_1991_2020_mean_c"] = baseline
        result["2024_mean_c"] = float(rows[-1]["Annual"])
        result["2024_anomaly_c"] = result["2024_mean_c"] - baseline
    results[kind] = result
print(json.dumps(results, indent=2))


{
  "rainfall": {
    "rows": 124,
    "columns": 23,
    "year_range": [
      1901,
      2024
    ],
    "duplicate_years": 0,
    "missing_years": [],
    "empty_cells": 0,
    "nonfinite_values": 0,
    "source_comparison_different_years": [],
    "source_numeric_values_compared": 2108,
    "annual_sum_discrepancies": [
      {
        "year": 1901,
        "annual_minus_months_mm": 0.1
      },
      {
        "year": 1902,
        "annual_minus_months_mm": 0.0
      },
      {
        "year": 1903,
        "annual_minus_months_mm": 0.1
      },
      {
        "year": 1904,
        "annual_minus_months_mm": -0.1
      },
      {
        "year": 1905,
        "annual_minus_months_mm": 0.1
      },
      {
        "year": 1906,
        "annual_minus_months_mm": 0.1
      },
      {
        "year": 1907,
        "annual_minus_months_mm": 0.0
      },
      {
        "year": 1908,
        "annual_minus_months_mm": 0.3
      },
      {
        "year": 1909,
        "annual_minus_mont